In [0]:
%sql
    
-- Aggregate CLM data by all perfect and near-match dimensions to CF table, store in temp table
CREATE OR REPLACE TEMPORARY VIEW clm_agg_temp AS
WITH base_agg AS (
SELECT
  -- HCC type (PH, OP, IP)
  clm.hcc,
  -- Perfect matches
  clm.service_code,                        -- = CF.HCEPRCAT
  clm.market_fnl,                           -- = CF.Market
  clm.tfm_product_fnl,                      -- = CF.TFM_Product
  clm.segment_name_fnl,                     -- = CF.Segment
  -- Pass-through ENROLL_KEYS (not in CF, carried for downstream grains)
  clm.global_cap,
  clm.drug_cov_type_fnl,
  -- Near matches
  clm.tfm_product_new_fnl,                  -- = CF.TFM_Product_New (89% overlap, missing DUAL_CHRONIC)
  clm.migration_source,                      -- = CF.Migration_Source
  clm.region_fnl,                            -- = CF.Region
  clm.region1_fnl,                           -- = CF.Region1
  clm.product_level_1_fnl,                   -- = CF.Product_Level_1
  clm.product_level_2_fnl,                   -- = CF.Product_Level_2
  clm.product_level_3_fnl,                   -- = CF.Product_Level_3
  clm.plan_level_1_fnl,                      -- = CF.Plan_Level_1
  clm.plan_level_2_fnl,                      -- = CF.Plan_Level_2
  clm.group_ind_fnl,                         -- = CF.Group_IND
  clm.delegated_entity,                      -- = CF.Delegated_Entity
  clm.erickson_desc_fnl,                     -- = CF.Erickson_Site
  clm.tfm_include_flag,
  clm.fin_submarket,                         -- = CF.Submarket
  clm.fin_scc,                               -- = CF.SCC
  clm.contractpbp_fnl,                       -- = CF.ContractPBP
  clm.ccreconadjmkt_fnl,                     -- = CF.CCReconAdjMkt
  clm.special_network,                       -- = CF.Special_Network
  clm.fin_inc_month,                        -- = CF.MTH (76% overlap, CLM has 2022 + Jul 2026 not in CF)
  -- Dollar measures
  SUM(clm.sum_allw_amt)         AS sum_allw_amt,
  SUM(clm.sum_net_pd_amt)       AS sum_net_pd_amt,
  -- Unit measures
  SUM(clm.sum_tadm_units)       AS sum_tadm_units,
  SUM(clm.sum_visits)           AS sum_visits,
  SUM(clm.sum_srvc_unit_cnt)    AS sum_srvc_unit_cnt,
  SUM(clm.sum_adj_srvc_unit_cnt) AS sum_adj_srvc_unit_cnt,
  SUM(clm.sum_case_cnt)         AS sum_case_cnt,
  SUM(clm.sum_fst_visits)       AS sum_fst_visits,
  SUM(clm.sum_calc_tadm_procedures) AS sum_calc_tadm_procedures,
  SUM(clm.sum_tadm_hcta_util)   AS sum_tadm_hcta_util,
  SUM(clm.sum_admits)           AS sum_admits,
  SUM(clm.sum_qtydays)          AS sum_qtydays
FROM prod_tadm.mr_cos_prod_actuarial.dev_tfm_hcta_mnr_clm_data clm
GROUP BY
  clm.hcc,
  clm.service_code,
  clm.market_fnl,
  clm.tfm_product_fnl,
  clm.segment_name_fnl,
  clm.global_cap,
  clm.drug_cov_type_fnl,
  clm.tfm_product_new_fnl,
  clm.migration_source,
  clm.region_fnl,
  clm.region1_fnl,
  clm.product_level_1_fnl,
  clm.product_level_2_fnl,
  clm.product_level_3_fnl,
  clm.plan_level_1_fnl,
  clm.plan_level_2_fnl,
  clm.group_ind_fnl,
  clm.delegated_entity,
  clm.erickson_desc_fnl,
  clm.tfm_include_flag,
  clm.fin_submarket,
  clm.fin_scc,
  clm.contractpbp_fnl,
  clm.ccreconadjmkt_fnl,
  clm.special_network,
  clm.fin_inc_month
)
-- Compute Duration on the ~54 distinct fin_inc_month values instead of
-- sorting 80M rows in a single partition when cell 3 materializes the view.
-- Duration 1 = most recent fin_inc_month (Jul 2026, no CF), 2 = Jun 2026, etc.
, mth_rank AS (
  SELECT fin_inc_month, DENSE_RANK() OVER (ORDER BY fin_inc_month DESC) AS Duration
  FROM (SELECT DISTINCT fin_inc_month FROM base_agg)
)
SELECT mth_rank.Duration, base_agg.*
FROM base_agg
JOIN mth_rank ON base_agg.fin_inc_month = mth_rank.fin_inc_month;

SELECT *
FROM clm_agg_temp
LIMIT 10;

In [0]:
%sql
-- CF summary: UNION ALL of PH, OP, IP CF tables with hcc tag, expanded to all matched dimensions
CREATE OR REPLACE TEMPORARY VIEW cf_summary_temp AS
WITH cf_all AS (
  -- PH (Physician / Professional Services)
  SELECT 'PH' AS hcc,
    HCEPRCAT, Market, TFM_Product, Segment, TFM_Product_New, Migration_Source,
    Region, Region1, Product_Level_1, Product_Level_2, Product_Level_3,
    Plan_Level_1, Plan_Level_2, Group_IND, Delegated_Entity, Erickson_Site,
    Submarket, SCC, ContractPBP, CCReconAdjMkt, Special_Network, MTH
    --Dollar measures
    ,UnCompleted_Allowed, Completed_Allowed
    ,UnCompleted_Net, Completed_Net,
    --Unit measures
    UnCompleted_Procedures, Completed_Procedures,
    CAST(0 AS DOUBLE) AS UnCompleted_Units, CAST(0 AS DOUBLE) AS Completed_Units,
    CAST(0 AS DOUBLE) AS UnCompleted_Visits, CAST(0 AS DOUBLE) AS Completed_Visits,
    CAST(0 AS DOUBLE) AS UnCompleted_Admits, CAST(0 AS DOUBLE) AS Completed_Admits,
    CAST(0 AS DOUBLE) AS UnCompleted_Days, CAST(0 AS DOUBLE) AS Completed_Days
  FROM prod_tadm.mr_cos_prod_actuarial.Medicare_CF_forDrillDown_PrServ

  UNION ALL

  -- OP (Outpatient) — service category = HCEBKCAT; Uncompleted_Net has lowercase 'c'
  SELECT 'OP' AS hcc,
    CASE WHEN HCEBKCAT='MHCDOP' THEN 'MH' 
          WHEN HCEBKCAT='OP_HSP' THEN 'OP_HOSPICE' 
          ELSE HCEBKCAT END AS HCEPRCAT
    ,Market, TFM_Product, Segment, TFM_Product_New, Migration_Source,
    Region, Region1, Product_Level_1, Product_Level_2, Product_Level_3,
    Plan_Level_1, Plan_Level_2, Group_IND, Delegated_Entity, Erickson_Site,
    Submarket, SCC, ContractPBP, CCReconAdjMkt, Special_Network, MTH
    --Dollar measures
    ,UnCompleted_Allowed, Completed_Allowed
    ,Uncompleted_Net AS UnCompleted_Net, Completed_Net
    --Unit measures
    ,UnCompleted_Procedures, Completed_Procedures
    ,UnCompleted_Units, Completed_Units
    ,UnCompleted_Visits, Completed_Visits
    ,CAST(0 AS DOUBLE) AS UnCompleted_Admits, CAST(0 AS DOUBLE) AS Completed_Admits
    ,CAST(0 AS DOUBLE) AS UnCompleted_Days, CAST(0 AS DOUBLE) AS Completed_Days
  FROM prod_tadm.mr_cos_prod_actuarial.Medicare_CF_forDrillDown_OP

  UNION ALL

  -- IP (Inpatient)
  SELECT 'IP' AS hcc,
    HCEPRCAT, Market, TFM_Product, Segment, TFM_Product_New, Migration_Source,
    Region, Region1, Product_Level_1, Product_Level_2, Product_Level_3,
    Plan_Level_1, Plan_Level_2, Group_IND, Delegated_Entity, Erickson_Site,
    Submarket, SCC, ContractPBP, CCReconAdjMkt, Special_Network, MTH
    --Dollar measures
    ,UnCompleted_Allowed, Completed_Allowed
    ,UnCompleted_Net, Completed_Net
    --Unit measures
    ,CAST(0 AS DOUBLE) AS UnCompleted_Procedures
    ,CAST(0 AS DOUBLE) AS Completed_Procedures
    ,CAST(0 AS DOUBLE) AS UnCompleted_Units
    ,CAST(0 AS DOUBLE) AS Completed_Units
    ,CAST(0 AS DOUBLE) AS UnCompleted_Visits
    ,CAST(0 AS DOUBLE) AS Completed_Visits
    ,UnCompleted_Admits, Completed_Admits
    ,UnCompleted_Days, Completed_Days
  FROM prod_tadm.mr_cos_prod_actuarial.Medicare_CF_forDrillDown_IP
),
agg AS (
  SELECT
    hcc,
    HCEPRCAT,                                -- = CLM.service_code
    Market,                                  -- = CLM.market_fnl
    TFM_Product,                             -- = CLM.tfm_product_fnl
    Segment,                                 -- = CLM.segment_name_fnl
    TFM_Product_New,                         -- = CLM.tfm_product_new_fnl
    Migration_Source,                         -- = CLM.migration_source
    Region,                                  -- = CLM.region_fnl
    Region1,                                 -- = CLM.region1_fnl
    Product_Level_1,                         -- = CLM.product_level_1_fnl
    Product_Level_2,                         -- = CLM.product_level_2_fnl
    Product_Level_3,                         -- = CLM.product_level_3_fnl
    Plan_Level_1,                            -- = CLM.plan_level_1_fnl
    Plan_Level_2,                            -- = CLM.plan_level_2_fnl
    Group_IND,                               -- = CLM.group_ind_fnl
    Delegated_Entity,                        -- = CLM.delegated_entity
    Erickson_Site,                           -- = CLM.erickson_desc_fnl
    Submarket,                               -- = CLM.fin_submarket
    SCC,                                     -- = CLM.fin_scc
    ContractPBP,                             -- = CLM.contractpbp_fnl
    CCReconAdjMkt,                           -- = CLM.ccreconadjmkt_fnl
    Special_Network,                         -- = CLM.special_network
    MTH,                                     -- = CLM.fin_inc_month
    -- Completion factor ratios
    TRY_DIVIDE(SUM(UnCompleted_Allowed), SUM(Completed_Allowed)) AS CF_Allowed,
    TRY_DIVIDE(SUM(UnCompleted_Net), SUM(Completed_Net)) AS CF_Net,
    -- Unit completion factor ratios
    TRY_DIVIDE(SUM(UnCompleted_Procedures), SUM(Completed_Procedures)) AS CF_Procedures,
    TRY_DIVIDE(SUM(UnCompleted_Units), SUM(Completed_Units)) AS CF_Units,
    TRY_DIVIDE(SUM(UnCompleted_Visits), SUM(Completed_Visits)) AS CF_Visits,
    TRY_DIVIDE(SUM(UnCompleted_Admits), SUM(Completed_Admits)) AS CF_Admits,
    TRY_DIVIDE(SUM(UnCompleted_Days), SUM(Completed_Days)) AS CF_Days,
    -- Component totals
    SUM(UnCompleted_Allowed) AS UnCompleted_Allowed,
    SUM(Completed_Allowed)   AS Completed_Allowed,
    SUM(UnCompleted_Net)     AS UnCompleted_Net,
    SUM(Completed_Net)       AS Completed_Net,
    -- Unit measures
    SUM(UnCompleted_Procedures) AS UnCompleted_Procedures,
    SUM(Completed_Procedures)   AS Completed_Procedures,
    SUM(UnCompleted_Units)      AS UnCompleted_Units,
    SUM(Completed_Units)        AS Completed_Units,
    SUM(UnCompleted_Visits)     AS UnCompleted_Visits,
    SUM(Completed_Visits)       AS Completed_Visits,
    SUM(UnCompleted_Admits)     AS UnCompleted_Admits,
    SUM(Completed_Admits)       AS Completed_Admits,
    SUM(UnCompleted_Days)       AS UnCompleted_Days,
    SUM(Completed_Days)         AS Completed_Days
  FROM cf_all
  GROUP BY hcc, HCEPRCAT, Market, TFM_Product, Segment, TFM_Product_New, Migration_Source,
    Region, Region1, Product_Level_1, Product_Level_2, Product_Level_3,
    Plan_Level_1, Plan_Level_2, Group_IND, Delegated_Entity, Erickson_Site,
    Submarket, SCC, ContractPBP, CCReconAdjMkt, Special_Network, MTH
)
-- Compute Duration on the ~54 distinct MTH values instead of sorting 62M rows.
-- CF Duration = DENSE_RANK(MTH DESC) + 1; the +1 aligns with CLM whose
-- Duration 1 = most recent month (Jul 2026, no CF), Duration 2 = Jun 2026 = CF's newest.
, mth_rank AS (
  SELECT MTH, DENSE_RANK() OVER (ORDER BY MTH DESC) + 1 AS Duration
  FROM (SELECT DISTINCT MTH FROM agg)
)
SELECT mth_rank.Duration, agg.*
FROM agg
JOIN mth_rank ON agg.MTH = mth_rank.MTH;

-- Preview
SELECT *
FROM cf_summary_temp
ORDER BY Duration ASC, hcc, Market, HCEPRCAT
LIMIT 10;

In [0]:
%sql
    
CREATE OR REPLACE TEMPORARY VIEW clm_cf_joined_temp AS
SELECT
  clm.*,
  cf.cf_Allowed,
  cf.cf_Net,
  -- Apply completion factors to dollar amounts
  clm.sum_allw_amt  / COALESCE(cf.CF_Allowed, 1) AS completed_allw_amt,
  clm.sum_net_pd_amt / COALESCE(cf.CF_Net, 1)    AS completed_net_pd_amt
  -- Apply completion factors to OP units
  ,case when clm.hcc='OP' 
    then clm.sum_tadm_hcta_util / COALESCE(cf.CF_Units, 1) 
    else 0 end as completed_OP_Units
  -- Apply completion factors to PH units
  ,case when clm.hcc='PH' 
    then clm.sum_tadm_units  / COALESCE(cf.CF_Procedures, 1) 
    else 0 end as completed_PH_Procedures
  -- Apply completion factors to IP units
  ,case when clm.hcc='IP' 
    then clm.sum_tadm_units  / COALESCE(cf.CF_Days, 1) 
    else 0 end as completed_IP_Days
  ,case when clm.hcc='IP' 
    then clm.sum_visits  / COALESCE(cf.CF_Admits, 1) 
    else 0 end as completed_IP_Admits
FROM clm_agg_temp clm
LEFT JOIN cf_summary_temp cf
  ON  clm.hcc                                 = cf.hcc
  AND COALESCE(clm.service_code, 'NA')       = COALESCE(cf.HCEPRCAT, 'NA')
  AND clm.Duration                             = cf.Duration
  AND COALESCE(clm.market_fnl, 'NA')          = COALESCE(cf.Market, 'NA')
  AND COALESCE(clm.tfm_product_fnl, 'NA')     = COALESCE(cf.TFM_Product, 'NA')
  AND COALESCE(clm.segment_name_fnl, 'NA')    = COALESCE(cf.Segment, 'NA')
  AND COALESCE(clm.tfm_product_new_fnl, 'NA') = COALESCE(cf.TFM_Product_New, 'NA')
  AND COALESCE(clm.migration_source, 'NA')    = COALESCE(cf.Migration_Source, 'NA')
  AND COALESCE(clm.region_fnl, 'NA')          = COALESCE(cf.Region, 'NA')
  AND COALESCE(clm.region1_fnl, 'NA')         = COALESCE(cf.Region1, 'NA')
  AND COALESCE(clm.product_level_1_fnl, 'NA') = COALESCE(cf.Product_Level_1, 'NA')
  AND COALESCE(clm.product_level_2_fnl, 'NA') = COALESCE(cf.Product_Level_2, 'NA')
  AND COALESCE(clm.product_level_3_fnl, 'NA') = COALESCE(cf.Product_Level_3, 'NA')
  AND COALESCE(clm.plan_level_1_fnl, 'NA')    = COALESCE(cf.Plan_Level_1, 'NA')
  AND COALESCE(clm.plan_level_2_fnl, 'NA')    = COALESCE(cf.Plan_Level_2, 'NA')
  AND COALESCE(clm.group_ind_fnl, 'NA')       = COALESCE(cf.Group_IND, 'NA')
  AND COALESCE(clm.delegated_entity, 'NA')    = COALESCE(cf.Delegated_Entity, 'NA')
  AND COALESCE(clm.erickson_desc_fnl, 'NA')   = COALESCE(cf.Erickson_Site, 'NA')
  AND COALESCE(clm.fin_submarket, 'NA')       = COALESCE(cf.Submarket, 'NA')
  AND COALESCE(clm.fin_scc, 'NA')             = COALESCE(cf.SCC, 'NA')
  AND COALESCE(clm.contractpbp_fnl, 'NA')     = COALESCE(cf.ContractPBP, 'NA')
  AND COALESCE(clm.ccreconadjmkt_fnl, 'NA')   = COALESCE(cf.CCReconAdjMkt, 'NA')
  AND COALESCE(clm.special_network, 'NA')     = COALESCE(cf.Special_Network, 'NA');

-- Preview top 10 rows
SELECT * FROM clm_cf_joined_temp
where Duration<>1
ORDER BY fin_inc_month DESC, market_fnl, service_code
LIMIT 10

In [0]:
%sql
--ensure that the majority of each duration is receiving a CF
-- % of total dollars with NULL CFs by Duration (excluding Duration 1 and >38)
SELECT
  t.Duration,t.HCC,
  ROUND(t.null_cf_allw_amt * 100.0 / NULLIF(t.total_allw_amt, 0), 4) AS pct_allw_null_cf,
  ROUND(t.null_cf_net_pd_amt * 100.0 / NULLIF(t.total_net_pd_amt, 0), 4) AS pct_net_null_cf,
  t.total_allw_amt,
  t.total_net_pd_amt,
  t.null_cf_allw_amt,
  t.null_cf_net_pd_amt
FROM (
  SELECT
    Duration,HCC,
    SUM(sum_allw_amt) AS total_allw_amt,
    SUM(sum_net_pd_amt) AS total_net_pd_amt,
    SUM(CASE WHEN cf_Allowed IS NULL THEN sum_allw_amt ELSE 0 END) AS null_cf_allw_amt,
    SUM(CASE WHEN cf_Net IS NULL THEN sum_net_pd_amt ELSE 0 END) AS null_cf_net_pd_amt
  FROM clm_cf_joined_temp
  WHERE Duration > 1 AND Duration <= 38
  GROUP BY Duration,HCC
) t
ORDER BY Duration,HCC

In [0]:
%sql
SELECT *
FROM clm_cf_joined_temp
WHERE Duration = 2
  AND cf_Allowed IS NULL
ORDER BY sum_allw_amt DESC

In [0]:
%sql
-- Total completed allowed and net dollars by month and Duration, with member counts
SELECT
  clm.HCC,
  clm.service_code,
  clm.Duration,
  clm.fin_inc_month,
  SUM(clm.completed_allw_amt)/mbr.total_member_cnt  AS comp_allw_PMPM,
  SUM(clm.completed_net_pd_amt)/mbr.total_member_cnt AS comp_net_PMPM,
  SUM(clm.completed_OP_Units)/mbr.total_member_cnt*12000  AS comp_OP_UtilPerK,
  SUM(clm.completed_PH_Procedures)/mbr.total_member_cnt*12000  AS comp_PH_Procedures_UtilPerK,
  SUM(clm.completed_IP_Days)/mbr.total_member_cnt*12000  AS comp_IP_Days_UtilPerK,
  SUM(clm.completed_IP_Admits)/mbr.total_member_cnt*12000  AS comp_IP_Admits_UtilPerK,
  mbr.total_member_cnt
FROM clm_cf_joined_temp clm
LEFT JOIN (
  SELECT fin_inc_month, SUM(sum_member_cnt) AS total_member_cnt
  FROM prod_tadm.mr_cos_prod_actuarial.dev_tfm_hcta_mnr_mbr_data
  WHERE migration_source NOT IN ('OAH')
    and coalesce(tfm_product_new_fnl,'UNK') <>'INSTITUTIONAL'
    and coalesce(fin_risk_type,'UNK')<>'GLOBAL'
  GROUP BY fin_inc_month
) mbr ON clm.fin_inc_month = mbr.fin_inc_month
WHERE clm.migration_source NOT IN ('OAH') 
  and clm.tfm_include_flag = 1
  and coalesce(clm.tfm_product_new_fnl,'UNK') <>'INSTITUTIONAL'
  and clm.Duration between 2 and 31
GROUP BY clm.HCC,clm.service_code,clm.Duration, clm.fin_inc_month, mbr.total_member_cnt
ORDER BY clm.HCC,clm.service_code,clm.Duration, clm.fin_inc_month DESC

In [0]:
%sql
-- Monthly summary of CLM+CF joined data by key dimensions
CREATE OR REPLACE TEMPORARY VIEW clm_cf_monthly_summary AS
SELECT
  Duration,
  fin_inc_month,
  hcc,
  service_code,
  global_cap,
  market_fnl,
  tfm_product_fnl,
  tfm_product_new_fnl,
  segment_name_fnl,
  drug_cov_type_fnl,
  -- Completed dollar totals
  SUM(completed_allw_amt)      AS completed_allw_amt,
  SUM(completed_net_pd_amt)    AS completed_net_pd_amt,
  -- Completed unit totals
  SUM(completed_OP_Units)      AS completed_OP_Units,
  SUM(completed_PH_Procedures) AS completed_PH_Procedures,
  SUM(completed_IP_Days)       AS completed_IP_Days,
  SUM(completed_IP_Admits)     AS completed_IP_Admits,
  -- Row count
  COUNT(*)                     AS row_cnt
FROM clm_cf_joined_temp
GROUP BY
  Duration,
  fin_inc_month,
  hcc,
  service_code,
  global_cap,
  market_fnl,
  tfm_product_fnl,
  tfm_product_new_fnl,
  segment_name_fnl,
  drug_cov_type_fnl
ORDER BY Duration, fin_inc_month DESC, hcc, service_code, market_fnl;

SELECT * FROM clm_cf_monthly_summary LIMIT 20;

In [0]:
%sql
CREATE OR REPLACE TABLE prod_tadm.mr_cos_prod_actuarial.dev_tfm_hcta_mnr_clm_completed AS
SELECT * FROM clm_cf_monthly_summary;

In [0]:
%sql
-- Membership summary by the same categoricals as clm_cf_monthly_summary
-- (hcc and service_code are claims-specific, so excluded)
CREATE OR REPLACE TEMPORARY VIEW mbr_monthly_summary AS
WITH base_agg AS (
  SELECT
    fin_inc_month,
    global_cap,
    market_fnl,
    tfm_product_fnl,
    tfm_product_new_fnl,
    segment_name_fnl,
    drug_cov_type_fnl,
    SUM(sum_member_cnt) AS sum_member_cnt
  FROM prod_tadm.mr_cos_prod_actuarial.dev_tfm_hcta_mnr_mbr_data
  GROUP BY
    fin_inc_month,
    global_cap,
    market_fnl,
    tfm_product_fnl,
    tfm_product_new_fnl,
    segment_name_fnl,
    drug_cov_type_fnl
),
mth_rank AS (
  SELECT fin_inc_month,
         DENSE_RANK() OVER (ORDER BY fin_inc_month DESC) AS Duration
  FROM (SELECT DISTINCT fin_inc_month FROM base_agg)
)
SELECT
  mr.Duration,
  ba.*
FROM base_agg ba
JOIN mth_rank mr ON ba.fin_inc_month = mr.fin_inc_month;

SELECT * FROM mbr_monthly_summary
ORDER BY Duration, fin_inc_month DESC, market_fnl
LIMIT 20;

In [0]:
%sql
CREATE OR REPLACE TABLE prod_tadm.mr_cos_prod_actuarial.dev_tfm_hcta_mnr_mbr_summary AS
SELECT * FROM mbr_monthly_summary;

## NULL CF Root Cause Analysis

**Investigated the 22-column join between `clm_agg_temp` and `cf_summary_temp` to identify why certain CLM rows receive no CF match.**

### Culprit #1: `fin_scc` NULL → guaranteed mismatch (\~88% of null-CF rows)
- **CF side** (`SCC`): **Never NULL**, never 'NA'. All 3,183 distinct values are 5-digit numeric codes (01000–99999).
- **CLM side** (`fin_scc`): Many rows have `NULL`. The join uses `COALESCE(clm.fin_scc, 'NA') = COALESCE(cf.SCC, 'NA')` — but since CF SCC is never NULL, the COALESCE on the CF side always resolves to the real SCC code, which never equals 'NA'.
- **Result**: Every CLM row with `fin_scc = NULL` is **guaranteed to fail the join** on this dimension alone, regardless of all other columns.

### Culprit #2: `tfm_product_new_fnl` = 'DUAL_CHRONIC' — not in CF
- **CF side** (`TFM_Product_New`): 9 distinct values — PPO, PFFS_IND, C&S DUAL, NPPO, CHRONIC, HMO, INSTITUTIONAL, DUAL, PEOPLES HEALTH.
- **CLM side** (`tfm_product_new_fnl`): Includes 'DUAL_CHRONIC' (noted at 89% overlap in cell 1 comments), which has **no matching CF value**.
- **Result**: CLM rows with `tfm_product_new_fnl = 'DUAL_CHRONIC'` will always fail if their value doesn't map to one of the 9 CF values.

### Culprit #3: Combinatorial sparsity at specific months
- Even when **all 22 individual dimension values exist** on both sides, the specific full combination may not exist in CF for the CLM row's month.
- Example: `IP_SWGBED / TX / NPPO / SCC=45930 / H2001-869-000 / OEB015300` exists at MTH=202605 but **not** at MTH=202606.
- This affects the \~12% of null-CF rows that have non-null `fin_scc` and valid `TFM_Product_New`.

### Impact by HCC (from Cell 4 results, Duration 2)
| HCC | Null-CF % of Allowed $ | Primary driver |
|-----|------------------------|----------------|
| IP  | \~0.94%                | fin_scc NULL + sparse combinations |
| OP  | \~0.50%                | fin_scc NULL |
| PH  | \~0.10%                | fin_scc NULL |

### Do the two culprits correlate? **No — they are independent populations.**
- Scanned the **top 292 null-CF rows** (by allowed $) at Duration 2: **every single one** has a valid `tfm_product_new_fnl` (CHRONIC, DUAL, HMO, PPO, NPPO, etc.) — zero `DUAL_CHRONIC` entries.
- This means **`fin_scc` NULL** drives the highest-dollar null CFs on rows that would otherwise match on TFM_Product_New.
- **`DUAL_CHRONIC`** rows ($1.58B / 1.71% of CLM allowed) are a separate, lower-dollar-per-row population that fails solely because the value doesn't exist in CF. They do not overlap with the `fin_scc` NULL rows in the high-impact tier.
- The two fixes are **additive**: resolving one addresses a completely different set of rows than resolving the other.

### Dimensions with full overlap (not causing mismatches)
`service_code / HCEPRCAT`, `market_fnl / Market`, `tfm_product_fnl / TFM_Product` (all 14 values shared), `segment_name_fnl / Segment`, `migration_source / Migration_Source` (CIP, CSP, MEDICA, NA, OAH, PC), `fin_inc_month / MTH` (202209–202606 shared), `region_fnl / Region`, `region1_fnl / Region1`, all Product_Level, Plan_Level, Group_IND, Delegated_Entity, Erickson_Site, Submarket (incl. UNK), CCReconAdjMkt, Special_Network (incl. OTH, ATL/OEB prefixes).
